In [16]:
%reset -f

In [17]:
from tensorflow import keras
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import os
from datetime import datetime

from sklearn.metrics import mean_squared_error, mean_absolute_error

import sys
sys.path.append(r"C:\ThesisWork\offical_approach\mth_project\mth_project\industrial_network_analysis")
from data_preprocessing import Dataset
from data_utils import remove_outliers

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

In [18]:
### READ DATA
device_name = "A1-RBX-243"
df = Dataset(f'C:\\ThesisWork\\offical_approach\\mth_project\\mth_project\\industrial_network_analysis\\Data082025\\{device_name}.csv')

# names for traffic on ports - BITS SENT/RECEIVED
bits = "bits"
bits_exclude = "unused"

def get_column_names_exclude(df, search_keyword, exclude_keyword):
    df_column_names = df.get_column_names(search_keyword)
    df_column_names_excluded = []

    for col in df_column_names:
        if exclude_keyword.lower() not in col.lower():
            df_column_names_excluded.append(col)

    return df_column_names_excluded


df_columns_bits_names = get_column_names_exclude(df, bits, bits_exclude)

df_bits = df.get_column_values(df_columns_bits_names)
df_bits.columns = df_bits.columns.str.replace('Interface Gi', '')
df_bits.columns = df_bits.columns.str.replace('Valmet SUPV switch connection', '')
df_bits.columns = df_bits.columns.str.replace('Supervisory access', '')
df_bits.columns = df_bits.columns.str.replace('Bits sent', 'out')
df_bits.columns = df_bits.columns.str.replace('Bits received', 'in')
df_bits.columns = df_bits.columns.str.replace('Valmet DNA', '')

df_bits = remove_outliers(df_bits, 3).dropna()
df_bits = df_bits[:24*60]

df_bits_train, df_bits_test = np.split(df_bits, [int(0.8 * len(df_bits))])

c:\ThesisWork\offical_approach\mth_project\.venv\Lib\site-packages\numpy\_core\fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


In [19]:
# LSTM forecasting program

### 1. Data Preparation

from sklearn.preprocessing import RobustScaler

scaler = RobustScaler()

# fit scaler only on training data
df_bits_train_scaled = pd.DataFrame(
    scaler.fit_transform(df_bits_train),
    columns=df_bits_train.columns,
    index=df_bits_train.index
)

# apply same transformation to test data
df_bits_test_scaled = pd.DataFrame(
    scaler.transform(df_bits_test),
    columns=df_bits_test.columns,
    index=df_bits_test.index
)

# divide data to train and test sets

# use previous 60 minutes to predict specified horizon
context_length = 60
X_train, y_train = [], []

for i in range(context_length, len(df_bits_train_scaled)):
    X_train.append(df_bits_train_scaled.iloc[i-context_length:i].values)
    y_train.append(df_bits_train_scaled.iloc[i].values)

X_train = np.array(X_train)
y_train = np.array(y_train)

### 2. Create Model

def create_online_multivariate_model(df,
                                     context_length=60,
                                     first_layer_units=64,
                                     second_layer_units=64,
                                     dense_units=128,
                                     activation='relu',
                                     dropout_rate=0.5):
    num_features = len(df.columns)
    
    model = keras.models.Sequential([
        keras.layers.LSTM(first_layer_units, return_sequences=True, input_shape=(context_length, num_features)),
        keras.layers.LSTM(second_layer_units, return_sequences=False),
        keras.layers.Dense(dense_units, activation=activation),
        keras.layers.Dropout(dropout_rate),
        keras.layers.Dense(len(df.columns))  # Output for each target variable
    ])

    model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001), 
                  loss='mse', 
                  metrics=['mae'])
    
    return model

model = create_online_multivariate_model(
    df=df_bits_train,
    context_length=context_length,
    first_layer_units=64,
    second_layer_units=64,
    dense_units=128,
    activation='relu',
    dropout_rate=0.5
)

# Train model

import time
start_time = time.time()

# Add early stopping to prevent overfitting
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)
]

history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,         
    validation_split=0.2,  
    callbacks=callbacks,
    verbose=1
)

end_time = time.time()
training_time = end_time - start_time
print(f"Training completed in {training_time:.2f} seconds")

### 3. Forecasting

X_test, y_test = [], []

# Need to combine end of training data with test data for proper sequencing
combined_data = pd.concat([df_bits_train_scaled.tail(context_length), df_bits_test_scaled])

for i in range(context_length, len(combined_data)):
    X_test.append(combined_data.iloc[i-context_length:i].values)
    y_test.append(combined_data.iloc[i].values)

X_test = np.array(X_test)
y_test = np.array(y_test)

start_time = time.time()
predictions_test = model.predict(X_test, verbose=0)
end_time = time.time()
prediction_time = end_time - start_time
actuals_test = y_test

print(f"Prediction completed in {prediction_time:.2f} seconds for {len(X_test)} samples")

### 4. Denormalization

# inverse transform predictions and actuals back to original scale
predictions_original = scaler.inverse_transform(predictions_test)
actuals_original = scaler.inverse_transform(actuals_test)

# create DataFrames with proper index (matching test data)
predictions_df = pd.DataFrame(
    data=predictions_original,
    columns=df_bits.columns,
    index=df_bits_test.index
)

actuals_df = pd.DataFrame(
    data=actuals_original,
    columns=df_bits.columns,
    index=df_bits_test.index
)

### 5. Evaluation

def calculate_metrics(actual, predicted, variable_name):
    """Calculate comprehensive metrics for a single variable"""
    mae = mean_absolute_error(actual, predicted)
    mse = mean_squared_error(actual, predicted)
    rmse = np.sqrt(mse)

    return {
        'Variable': variable_name,
        'MAE': mae,
        'MSE': mse,
        'RMSE': rmse
    }

results = []
for col in predictions_df.columns:
    actual = actuals_df[col].values
    predicted = predictions_df[col].values

    metrics = calculate_metrics(actual, predicted, col)
    results.append(metrics)

results_df = pd.DataFrame(results)

c:\ThesisWork\offical_approach\mth_project\.venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 9s 136ms/step - loss: 0.6056 - mae: 0.4471 - val_loss: 0.4825 - val_mae: 0.3233 - learning_rate: 0.0010
Epoch 2/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 4s 150ms/step - loss: 0.4449 - mae: 0.3590 - val_loss: 0.4281 - val_mae: 0.2927 - learning_rate: 0.0010
Epoch 3/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 4s 122ms/step - loss: 0.3614 - mae: 0.3339 - val_loss: 0.3687 - val_mae: 0.2838 - learning_rate: 0.0010
Epoch 4/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 111ms/step - loss: 0.2752 - mae: 0.3208 - val_loss: 0.2721 - val_mae: 0.2700 - learning_rate: 0.0010
Epoch 5/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 4s 125ms/step - loss: 0.2095 - mae: 0.2951 - val_loss: 0.1951 - val_mae: 0.2478 - learning_rate: 0.0010
Epoch 6/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 119ms/step - loss: 0.1773 - mae: 0.2857 - val_loss: 0.1728 - val_mae: 0.2485 - learning_rate: 0.0010
Epoch 7/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 103ms/step - loss: 0.1547 - mae: 0.2721 - val_loss: 0.1499 - val_mae: 0.2308 - learning_rate: 0.0010
Epoch 

In [ ]:
for index, row in results_df.iterrows():
    print(f"Variable: {row['Variable']}")
    print(f"  MAE:  {row['MAE']:.4f}")
    print(f"  MSE:  {row['MSE']:.4f}")
    print(f"  RMSE: {row['RMSE']:.4f}")
    print(f" Results in percentage: {(row['MAE']/ (actuals_df[row['Variable']].max() - actuals_df[row['Variable']].min())) * 100:.2f}")
    print("-" * 30)

In [ ]:
# visualization

def plot_single_variable_forecast(variable_idx, train_data_normalized, test_data_denormalized, 
                                 lstm_forecast_denormalized, scaler):
    if variable_idx >= len(lstm_forecast_denormalized.columns):
        print(f"Variable index {variable_idx} out of range. Max index: {len(lstm_forecast_denormalized.columns)-1}")
        return
    
    col = lstm_forecast_denormalized.columns[variable_idx]
    
    train_denormalized = pd.DataFrame(
        scaler.inverse_transform(train_data_normalized),
        columns=train_data_normalized.columns,
        index=train_data_normalized.index
    )
    
    plt.figure(figsize=(15, 8))
    
    plt.plot(train_denormalized.index, train_denormalized[col], 
             label='Training Data', color='blue', alpha=0.7)
    
    plt.plot(test_data_denormalized.index, test_data_denormalized[col], 
             label='Actual', color='green', linewidth=2)

    plt.plot(lstm_forecast_denormalized.index, lstm_forecast_denormalized[col], 
             label='LSTM Forecast', color='red', linewidth=2, linestyle='--', marker='s', markersize=3)

    plt.axvline(x=test_data_denormalized.index[0], color='gray', 
                linestyle=':', alpha=0.7, label='Forecast Start')
    
    plt.title(f'LSTM Forecast vs Actual - {col}', fontsize=14)
    plt.xlabel('Time')
    plt.ylabel('Value')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


In [ ]:
plot_single_variable_forecast(
    variable_idx=4,
    train_data_normalized=df_bits_train_scaled,
    test_data_denormalized=actuals_df,
    lstm_forecast_denormalized=predictions_df,
    scaler=scaler
)

In [ ]:
import pandas as pd
device_name="SW-SUPV-243"
data_path="C:\\ThesisWork\\offical_approach\\mth_project\\mth_project\\industrial_network_analysis\\Data082025\\"

df_removed_nans_forecasting = pd.read_csv(f'{data_path}{device_name}_forecasting.csv')
df_removed_nans_classification = pd.read_csv(f'{data_path}{device_name}_statuses.csv')